In [1]:
# Step 1. Import dependencies
import pandas as pd

# File paths (modify if needed)
rrt_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_concepts_exports/rrt.csv"
icustays_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports/icu_icustays.csv"

# Load datasets
rrt = pd.read_csv(rrt_path)
icustays = pd.read_csv(icustays_path)

print("rrt.csv shape:", rrt.shape)
print("icu_icustays.csv shape:", icustays.shape)

rrt.csv shape: (4098630, 5)
icu_icustays.csv shape: (94458, 8)


In [2]:
# Step 2. Inspect schemas
print("rrt.csv columns:", rrt.columns.tolist())
print("icu_icustays.csv columns:", icustays.columns.tolist())

# Quick look at first rows
display(rrt.head())
display(icustays.head())

rrt.csv columns: ['stay_id', 'charttime', 'dialysis_present', 'dialysis_active', 'dialysis_type']
icu_icustays.csv columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']


,stay_id,charttime,dialysis_present,dialysis_active,dialysis_type
0,30003130,2155-01-31 18:43:00,1,0,NaN
1,30003130,2155-01-31 20:20:00,1,0,NaN
2,30003130,2155-02-01 00:13:00,1,0,NaN
3,30003130,2155-02-01 04:45:00,1,0,NaN
4,30003130,2155-02-01 07:26:00,1,0,NaN


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


In [3]:
# Step 3. Merge RRT with ICU stays to get subject_id and hadm_id
dialysis = rrt.merge(
    icustays[["subject_id", "hadm_id", "stay_id"]],
    on="stay_id",
    how="left"
)

print("After merge:", dialysis.shape)
display(dialysis.head())

After merge: (4098630, 7)


,stay_id,charttime,dialysis_present,dialysis_active,dialysis_type,subject_id,hadm_id
0,30003130,2155-01-31 18:43:00,1,0,NaN,15855449,26043927
1,30003130,2155-01-31 20:20:00,1,0,NaN,15855449,26043927
2,30003130,2155-02-01 00:13:00,1,0,NaN,15855449,26043927
3,30003130,2155-02-01 04:45:00,1,0,NaN,15855449,26043927
4,30003130,2155-02-01 07:26:00,1,0,NaN,15855449,26043927


In [4]:
# Step 4. Select & rename columns
dialysis_final = dialysis.rename(columns={
    "subject_id": "pat_id",
    "hadm_id": "csn",
    "charttime": "service_timestamp"
})[[
    "csn", "pat_id", "service_timestamp",
    "dialysis_present", "dialysis_active", "dialysis_type"
]]

display(dialysis_final.head(10))

,csn,pat_id,service_timestamp,dialysis_present,dialysis_active,dialysis_type
0,26043927,15855449,2155-01-31 18:43:00,1,0,NaN
1,26043927,15855449,2155-01-31 20:20:00,1,0,NaN
2,26043927,15855449,2155-02-01 00:13:00,1,0,NaN
3,26043927,15855449,2155-02-01 04:45:00,1,0,NaN
4,26043927,15855449,2155-02-01 07:26:00,1,0,NaN
5,26043927,15855449,2155-02-01 07:58:00,1,0,NaN
6,26043927,15855449,2155-02-01 08:31:00,1,0,NaN
7,26043927,15855449,2155-02-01 09:07:00,1,0,IHD
8,26043927,15855449,2155-02-01 09:07:00,1,1,IHD
9,26043927,15855449,2155-02-01 12:07:00,1,0,NaN


In [5]:
# Step 5. Save the final DIALYSIS table
output_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/DIALYSIS.csv"
dialysis_final.to_csv(output_path, index=False)

print(f"✅ DIALYSIS table saved to {output_path}")


✅ DIALYSIS table saved to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/DIALYSIS.csv
